# Notebook 6 — Dashboard Interactivo · Plotly Dash

**Proyecto:** Portfolio Data Analyst — E-Commerce Analysis  
**Prerrequisito:** Notebooks 3 y 4 ejecutadas — esta notebook lee los CSVs de `../data/clean/`.

**Qué hace esta notebook:**
1. Instala las dependencias necesarias
2. Define los datos y cálculos base
3. Construye la app Dash completa con 4 páginas
4. Corre el servidor en `localhost:8050`

**Para usarlo:** ejecutar todas las celdas → abrir el browser en `http://localhost:8050`

---
## Índice
1. [Instalación de dependencias](#1)
2. [Carga de datos y cálculos base](#2)
3. [Paleta de colores y helpers](#3)
4. [Layout — estructura de las 4 páginas](#4)
5. [Callbacks — interactividad real](#5)
6. [Correr la app](#6)
7. [Cómo deployar en Render.com](#7)

---
## 1. Instalación de dependencias <a id='1'></a>

Si no tenés Dash instalado, esta celda lo instala. Solo necesita correr una vez.

In [1]:
import subprocess, sys

paquetes = ['dash', 'plotly', 'pandas', 'numpy']
for pkg in paquetes:
    try:
        __import__(pkg)
        print(f'  {pkg} ya instalado ✓')
    except ImportError:
        print(f'  Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  {pkg} instalado ✓')


  dash ya instalado ✓
  plotly ya instalado ✓
  pandas ya instalado ✓
  numpy ya instalado ✓


---
## 2. Carga de datos y cálculos base <a id='2'></a>

Todos los DataFrames y métricas se calculan una sola vez aquí. Los callbacks de Dash los usan
sin recalcular desde cero en cada interacción — solo filtran el dato ya procesado.

In [2]:
import pandas as pd
import numpy as np
from collections import Counter

CLEAN_DIR = '../data/clean/'

dv  = pd.read_csv(f'{CLEAN_DIR}dim_ventas.csv', encoding='utf-8-sig')
inv = pd.read_csv(f'{CLEAN_DIR}inventory_adjustments.csv', encoding='utf-8-sig')
env = pd.read_csv(f'{CLEAN_DIR}dim_envios.csv', encoding='utf-8-sig')

dv['InvoiceDate']  = pd.to_datetime(dv['InvoiceDate'])
inv['InvoiceDate'] = pd.to_datetime(inv['InvoiceDate'])
env['InvoiceDate'] = pd.to_datetime(env['InvoiceDate'])

for df_tmp in [dv, inv]:
    df_tmp['Year']  = df_tmp['InvoiceDate'].dt.year
    df_tmp['Month'] = df_tmp['InvoiceDate'].dt.month
dv['Hour'] = dv['InvoiceDate'].dt.hour

ventas    = dv[~dv['IsCancellation'] & (dv['UnitPrice'] > 0)].copy()
ventas_id = ventas[~ventas['IsGuest']].copy()
cancel    = dv[dv['IsCancellation']].copy()

UK_TICKET = (ventas_id[ventas_id['Country'] == 'United Kingdom']
             .groupby('InvoiceNo')['Revenue'].sum().mean())
UMBRAL_MAYOR = UK_TICKET * 3

cid_ticket = (ventas_id.groupby(['CustomerID', 'InvoiceNo'])['Revenue']
              .sum().reset_index()
              .groupby('CustomerID')['Revenue'].mean())

MAYOR_CIDS = set(cid_ticket[cid_ticket > UMBRAL_MAYOR].index.tolist())
RETAIL_CIDS = set(cid_ticket[cid_ticket <= UMBRAL_MAYOR].index.tolist())

ventas_id = ventas_id.copy()
ventas_id['Segment'] = ventas_id['CustomerID'].apply(
    lambda c: 'Mayorista' if c in MAYOR_CIDS else 'Retail'
)

KPI = {
    'rev_neto':     round(dv['Revenue'].sum(), 2),
    'rev_bruto':    round(ventas['Revenue'].sum(), 2),
    'n_facturas':   dv['InvoiceNo'].nunique(),
    'n_clientes':   ventas_id['CustomerID'].nunique(),
    'n_productos':  ventas['StockCode'].nunique(),
    'ticket_avg':   round(ventas.groupby('InvoiceNo')['Revenue'].sum().mean(), 2),
    'rev_envios':   round(env['Revenue'].sum(), 2),
    'pct_envios':   round(env['Revenue'].sum() / ventas['Revenue'].sum() * 100, 2),
    'n_paises':     ventas['Country'].nunique(),
    'n_mayor_cids': len(MAYOR_CIDS),
}

# Tasa de cancelación
cancel_by_mes = dv.groupby(['Year', 'Month']).apply(
    lambda x: pd.Series({
        'n_total':  x['InvoiceNo'].nunique(),
        'n_cancel': x.loc[x['IsCancellation'], 'InvoiceNo'].str[1:].nunique(),
    }), include_groups=False
).reset_index()
cancel_by_mes['tasa'] = (cancel_by_mes['n_cancel'] /
                          cancel_by_mes['n_total'] * 100).round(2)
cancel_by_mes['label'] = (cancel_by_mes['Month'].map(
    {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
     7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}) +
    ' ' + cancel_by_mes['Year'].astype(str).str[-2:])
cancel_by_mes = cancel_by_mes.sort_values(['Year','Month'])
KPI['tasa_cancel_media'] = round(cancel_by_mes['tasa'].mean(), 2)
KPI['cv_cancel'] = round(cancel_by_mes['tasa'].std() /
                         cancel_by_mes['tasa'].mean() * 100, 1)

rev_mensual = (ventas[(ventas['Year']==2011) & (ventas['Month']<=11)]
               .groupby('Month')['Revenue'].sum().reset_index())
rev_mensual['label'] = rev_mensual['Month'].map(
    {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
     7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'})
rev_mensual = rev_mensual.sort_values('Month')

rev_hora = ventas.groupby('Hour')['Revenue'].sum().reset_index()
rev_hora = rev_hora.sort_values('Hour')

rev_cid = (ventas_id.groupby('CustomerID')['Revenue']
           .sum().sort_values(ascending=False).reset_index())
rev_cid['acum_pct']  = rev_cid['Revenue'].cumsum() / rev_cid['Revenue'].sum() * 100
rev_cid['rank_pct']  = np.arange(1, len(rev_cid)+1) / len(rev_cid) * 100
rev_cid['Segment']   = rev_cid['CustomerID'].apply(
    lambda c: 'Mayorista' if c in MAYOR_CIDS else 'Retail'
)
top20_n   = int(len(rev_cid) * 0.20)
KPI['pareto_top20'] = round(rev_cid.head(top20_n)['Revenue'].sum() /
                            rev_cid['Revenue'].sum() * 100, 1)

top10_cid = rev_cid.head(10).copy()
top10_cid['label'] = top10_cid['CustomerID'].apply(lambda c: f'CID {int(c)}')

n_pedidos = ventas_id.groupby('CustomerID')['InvoiceNo'].nunique()
freq_dist = pd.DataFrame({
    'Rango':  ['1 pedido', '2-5 pedidos', '6-20 pedidos', '21+ pedidos'],
    'N':      [int((n_pedidos==1).sum()),
               int(((n_pedidos>=2)&(n_pedidos<=5)).sum()),
               int(((n_pedidos>=6)&(n_pedidos<=20)).sum()),
               int((n_pedidos>20).sum())],
})

sc_rev = (ventas.groupby('StockCode').agg(
    Revenue=('Revenue','sum'), Qty=('Quantity','sum')
).sort_values('Revenue', ascending=False).reset_index())
sc_rev['pct'] = (sc_rev['Revenue'] / sc_rev['Revenue'].sum() * 100).round(2)
sc_rev['acum'] = sc_rev['pct'].cumsum().round(2)
# Descripción canónica
NOTAS = ['wrongly','damaged','found','check','lost','missing','destroyed',
         'thrown','temp','error','sold as','wet','cracked','?','adjust']
desc_canon = {}
for sc, grp in ventas.groupby('StockCode')['Description']:
    reales = [d for d in grp.dropna() if not any(k in str(d).lower() for k in NOTAS)]
    if reales:
        desc_canon[sc] = Counter(reales).most_common(1)[0][0]
    else:
        desc_canon[sc] = 'Sin descripción'
sc_rev['Description'] = sc_rev['StockCode'].map(desc_canon).fillna('Sin descripción')
top10_sc  = sc_rev.head(10).copy()
top20_sc  = sc_rev.head(20).copy()

total_rev = sc_rev['Revenue'].sum()
conc_pts  = []
for n in [1,5,10,25,50,75,100,150,200,300,500,750,1000,1500,2000,3000,len(sc_rev)]:
    conc_pts.append({'n': n, 'pct': round(sc_rev.head(n)['Revenue'].sum()/total_rev*100, 2)})
conc_df = pd.DataFrame(conc_pts)

dv_sc    = set(dv['StockCode'].unique())
inv_excl = inv[~inv['StockCode'].isin(dv_sc)].copy()
bajas_m  = (inv_excl.groupby(['Year','Month']).size()
            .reset_index(name='n_bajas').sort_values(['Year','Month']))
bajas_m['label'] = (bajas_m['Month'].map(
    {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
     7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}) +
    ' ' + bajas_m['Year'].astype(str).str[-2:])

pais_stats = (ventas_id.groupby('Country').agg(
    Revenue  = ('Revenue',      'sum'),
    n_cli    = ('CustomerID',   'nunique'),
    n_fact   = ('InvoiceNo',    'nunique'),
).reset_index())
ticket_pais = (ventas_id.groupby(['Country','InvoiceNo'])['Revenue']
               .sum().reset_index().groupby('Country')['Revenue'].mean()
               .rename('ticket'))
pais_stats = pais_stats.merge(ticket_pais, on='Country')
pais_stats['ratio']    = (pais_stats['ticket'] / UK_TICKET).round(2)
pais_stats['IsMayor']  = pais_stats['ratio'] > 3
pais_stats['pct_rev']  = (pais_stats['Revenue'] / pais_stats['Revenue'].sum() * 100).round(2)
pais_stats             = pais_stats.sort_values('Revenue', ascending=False)

PAISES_LISTA = [{'label': ('★ ' if r['IsMayor'] else '') + r['Country'],
                 'value': r['Country']}
                for _, r in pais_stats.head(20).iterrows()]

for k, v in KPI.items():
    print(f'  {k}: {v}')
print(f'  n_meses_cancel: {len(cancel_by_mes)}')
print(f'  n_sc_activos: {len(sc_rev)}')
print(f'  n_paises: {len(pais_stats)}')

  rev_neto: 9770919.71
  rev_bruto: 10246820.87
  n_facturas: 23960
  n_clientes: 4334
  n_productos: 3791
  ticket_avg: 518.22
  rev_envios: 279462.12
  pct_envios: 2.73
  n_paises: 38
  n_mayor_cids: 102
  tasa_cancel_media: 14.53
  cv_cancel: 10.8
  pareto_top20: 74.6
  n_meses_cancel: 13
  n_sc_activos: 3791
  n_paises: 37


---
## 3. Paleta de colores y helpers <a id='3'></a>

In [3]:
import plotly.graph_objects as go

PAL = {
    'bg':      '#0d0f14', 'bg2': '#13161e', 'bg3': '#1a1e28',
    'text':    '#e8eaf0', 'text2': '#8b90a0', 'text3': '#555b6e',
    'azul':    '#4f8ef7', 'verde':  '#3ec987', 'rojo':   '#f0546a',
    'ocre':    '#f5a623', 'violeta':'#9b7ef8', 'cyan':   '#42d4f4',
    'border':  'rgba(255,255,255,0.07)',
}

def alpha(hex_color, a):
    h = hex_color.lstrip('#')
    r,g,b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'rgba({r},{g},{b},{a})'

def base_layout(**kwargs):
    return dict(
        paper_bgcolor = 'rgba(0,0,0,0)',
        plot_bgcolor  = 'rgba(0,0,0,0)',
        font          = dict(family='DM Sans, sans-serif', color=PAL['text2'], size=12),
        margin        = dict(l=10, r=10, t=30, b=10),
        xaxis         = dict(gridcolor=PAL['border'], linecolor=PAL['border'],
                             tickcolor=PAL['text3'], tickfont=dict(color=PAL['text2'])),
        yaxis         = dict(gridcolor=PAL['border'], linecolor=PAL['border'],
                             tickcolor=PAL['text3'], tickfont=dict(color=PAL['text2'])),
        showlegend    = False,
        **kwargs
    )

def fmt_gbp(v):
    if v >= 1e6: return f'£{v/1e6:.2f}M'
    if v >= 1e3: return f'£{v/1e3:.0f}K'
    return f'£{v:,.0f}'

def fmt_num(v):
    return f'{v:,}'.replace(',','.')


---
## 4. Layout — estructura de las 4 páginas <a id='4'></a>

El layout de Dash es declarativo — se define como un árbol de componentes.
Los callbacks de la siguiente celda le dan vida a los filtros.

In [4]:
from dash import Dash, dcc, html, Input, Output, callback

CSS_EXTERNO = [
    'https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap',
]

ESTILOS = {
    'app': {
        'background': PAL['bg'],
        'minHeight':  '100vh',
        'fontFamily': 'DM Sans, sans-serif',
    },
    'header': {
        'background':    PAL['bg2'],
        'borderBottom':  f'1px solid {PAL["border"]}',
        'padding':       '0 28px',
        'display':       'flex',
        'alignItems':    'center',
        'justifyContent':'space-between',
        'height':        '56px',
        'position':      'sticky',
        'top':           '0',
        'zIndex':        '100',
    },
    'brand': {
        'display': 'flex', 'alignItems': 'center', 'gap': '10px'
    },
    'dot': {
        'width':'8px','height':'8px','borderRadius':'50%',
        'background': PAL['azul'],
        'boxShadow':  f'0 0 8px {PAL["azul"]}',
    },
    'title': {
        'fontSize':'13px','fontWeight':'600','color': PAL['text'],
        'letterSpacing': '.03em',
    },
    'sub': {
        'fontSize':'11px','color': PAL['text3'],
        'fontFamily':'DM Mono, monospace','marginLeft':'6px',
    },
    'main': {
        'padding': '24px 28px', 'maxWidth': '1400px', 'margin': '0 auto'
    },
    'card': {
        'background':   PAL['bg2'],
        'border':       f'1px solid {PAL["border"]}',
        'borderRadius': '12px',
        'padding':      '18px',
        'marginBottom': '14px',
    },
    'card_title': {
        'fontSize':'11px','fontWeight':'600','color': PAL['text2'],
        'textTransform':'uppercase','letterSpacing':'.05em','marginBottom':'4px',
    },
    'card_sub': {
        'fontSize':'11.5px','color': PAL['text3'],'marginBottom':'12px',
    },
    'kpi_strip': {
        'display':'grid',
        'gridTemplateColumns':'repeat(6,1fr)',
        'gap':'10px','marginBottom':'20px',
    },
    'kpi': {
        'background':   PAL['bg2'],
        'border':       f'1px solid {PAL["border"]}',
        'borderRadius': '12px',
        'padding':      '14px 16px',
    },
    'kpi_label': {
        'fontSize':'10.5px','color': PAL['text3'],
        'letterSpacing':'.04em','textTransform':'uppercase','marginBottom':'5px',
    },
    'kpi_value': {
        'fontSize':'22px','fontWeight':'600','color': PAL['text'],
        'fontFamily':'DM Mono, monospace','lineHeight':'1',
    },
    'kpi_sub': {
        'fontSize':'11px','color': PAL['text3'],'marginTop':'4px',
    },
    'narrative': {
        'fontSize':'13px','color': PAL['text2'],'marginTop':'6px',
        'lineHeight':'1.6','maxWidth':'700px','padding':'10px 14px',
        'background': PAL['bg3'],'borderRadius':'8px',
        'borderLeft':f'3px solid {PAL["azul"]}','marginBottom':'18px',
    },
    'filter_row': {
        'display':'flex','gap':'10px','marginBottom':'16px',
        'alignItems':'center','flexWrap':'wrap',
    },
    'filter_label': {
        'fontSize':'11.5px','color': PAL['text3'],
    },
    'grid_2': {
        'display':'grid','gridTemplateColumns':'1fr 1fr','gap':'14px',
    },
    'grid_2_1': {
        'display':'grid','gridTemplateColumns':'2fr 1fr','gap':'14px',
    },
    'grid_1_2': {
        'display':'grid','gridTemplateColumns':'1fr 2fr','gap':'14px',
    },
    'footer': {
        'textAlign':'center','padding':'16px',
        'fontSize':'11px','color': PAL['text3'],
        'borderTop':f'1px solid {PAL["border"]}','marginTop':'30px',
    },
    'tab_style': {
        'background': PAL['bg2'],
        'border':     f'1px solid {PAL["border"]}',
        'color':      PAL['text2'],
        'padding':    '8px 20px',
        'fontSize':   '13px',
        'fontFamily': 'DM Sans, sans-serif',
    },
    'tab_selected': {
        'background':  alpha(PAL['azul'], 0.15),
        'border':      f'1px solid {PAL["border"]}',
        'borderBottom':f'2px solid {PAL["azul"]}',
        'color':       PAL['azul'],
        'padding':     '8px 20px',
        'fontSize':    '13px',
        'fontFamily':  'DM Sans, sans-serif',
        'fontWeight':  '600',
    },
}

def kpi_card(label, value, sub='', badge=None, badge_color=PAL['azul']):
    children = [
        html.Div(label, style=ESTILOS['kpi_label']),
        html.Div(value, style=ESTILOS['kpi_value']),
    ]
    if sub:
        children.append(html.Div(sub, style=ESTILOS['kpi_sub']))
    if badge:
        children.append(html.Span(badge, style={
            'display':'inline-block','fontSize':'10px','fontWeight':'500',
            'padding':'2px 8px','borderRadius':'20px','marginTop':'5px',
            'background':f'rgba({",".join(str(int(badge_color.lstrip("#")[i:i+2],16)) for i in (0,2,4))},0.15)',
            'color': badge_color,
        }))
    return html.Div(children, style=ESTILOS['kpi'])

def card(title, subtitle, children, extra_style=None):
    s = {**ESTILOS['card']}
    if extra_style:
        s.update(extra_style)
    return html.Div([
        html.Div(title,    style=ESTILOS['card_title']),
        html.Div(subtitle, style=ESTILOS['card_sub']),
        *children,
    ], style=s)


In [5]:
from dash import Dash, dcc, html, Input, Output, State, callback

CSS_EXTERNO = [
    'https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap',
]

ESTILOS = {
    'app': {'background': PAL['bg'], 'minHeight': '100vh', 'fontFamily': 'DM Sans, sans-serif'},
    'header': {
        'background': PAL['bg2'], 'borderBottom': f'1px solid {PAL["border"]}',
        'padding': '0 28px', 'display': 'flex', 'alignItems': 'center',
        'justifyContent': 'space-between', 'height': '56px',
        'position': 'sticky', 'top': '0', 'zIndex': '100',
    },
    'brand':  {'display': 'flex', 'alignItems': 'center', 'gap': '10px'},
    'dot':    {'width':'8px','height':'8px','borderRadius':'50%','background':PAL['azul'],'boxShadow':f'0 0 8px {PAL["azul"]}'},
    'title':  {'fontSize':'13px','fontWeight':'600','color':PAL['text'],'letterSpacing':'.03em'},
    'sub':    {'fontSize':'11px','color':PAL['text3'],'fontFamily':'DM Mono, monospace','marginLeft':'6px'},
    'main':   {'padding':'24px 28px','maxWidth':'1400px','margin':'0 auto'},
    'card':   {'background':PAL['bg2'],'border':f'1px solid {PAL["border"]}','borderRadius':'12px','padding':'18px','marginBottom':'14px'},
    'card_title': {'fontSize':'11px','fontWeight':'600','color':PAL['text2'],'textTransform':'uppercase','letterSpacing':'.05em','marginBottom':'4px'},
    'card_sub':   {'fontSize':'11.5px','color':PAL['text3'],'marginBottom':'12px'},
    'kpi_strip':  {'display':'grid','gridTemplateColumns':'repeat(6,1fr)','gap':'10px','marginBottom':'20px'},
    'kpi':    {'background':PAL['bg2'],'border':f'1px solid {PAL["border"]}','borderRadius':'12px','padding':'14px 16px'},
    'kpi_label':  {'fontSize':'10.5px','color':PAL['text3'],'letterSpacing':'.04em','textTransform':'uppercase','marginBottom':'5px'},
    'kpi_value':  {'fontSize':'22px','fontWeight':'600','color':PAL['text'],'fontFamily':'DM Mono, monospace','lineHeight':'1'},
    'kpi_sub':    {'fontSize':'11px','color':PAL['text3'],'marginTop':'4px'},
    'narrative':  {
        'fontSize':'13px','color':PAL['text2'],'marginTop':'6px','lineHeight':'1.6',
        'maxWidth':'700px','padding':'10px 14px','background':PAL['bg3'],'borderRadius':'8px',
        'borderLeft':f'3px solid {PAL["azul"]}','marginBottom':'18px',
    },
    'filter_row':   {'display':'flex','gap':'10px','marginBottom':'16px','alignItems':'center','flexWrap':'wrap'},
    'filter_label': {'fontSize':'11.5px','color':PAL['text3']},
    'grid_2':   {'display':'grid','gridTemplateColumns':'1fr 1fr','gap':'14px'},
    'grid_2_1': {'display':'grid','gridTemplateColumns':'2fr 1fr','gap':'14px'},
    'grid_1_2': {'display':'grid','gridTemplateColumns':'1fr 2fr','gap':'14px'},
    'footer':   {'textAlign':'center','padding':'16px','fontSize':'11px','color':PAL['text3'],'borderTop':f'1px solid {PAL["border"]}','marginTop':'30px'},
    'tab_style': {'background':PAL['bg2'],'border':f'1px solid {PAL["border"]}','color':PAL['text2'],'padding':'8px 20px','fontSize':'13px','fontFamily':'DM Sans, sans-serif'},
    'tab_selected': {
        'background':alpha(PAL['azul'],0.15),'border':f'1px solid {PAL["border"]}',
        'borderBottom':f'2px solid {PAL["azul"]}','color':PAL['azul'],
        'padding':'8px 20px','fontSize':'13px','fontFamily':'DM Sans, sans-serif','fontWeight':'600',
    },
}

def kpi_card(label, value, sub='', badge=None, badge_color=PAL['azul']):
    children = [html.Div(label, style=ESTILOS['kpi_label']), html.Div(value, style=ESTILOS['kpi_value'])]
    if sub:   children.append(html.Div(sub, style=ESTILOS['kpi_sub']))
    if badge: children.append(html.Span(badge, style={
        'display':'inline-block','fontSize':'10px','fontWeight':'500',
        'padding':'2px 8px','borderRadius':'20px','marginTop':'5px',
        'background':f'rgba({",".join(str(int(badge_color.lstrip("#")[i:i+2],16)) for i in (0,2,4))},0.15)',
        'color': badge_color,
    }))
    return html.Div(children, style=ESTILOS['kpi'])

def card(title, subtitle, children, extra_style=None):
    s = {**ESTILOS['card']}
    if extra_style: s.update(extra_style)
    return html.Div([html.Div(title, style=ESTILOS['card_title']),
                     html.Div(subtitle, style=ESTILOS['card_sub']), *children], style=s)

app = Dash(__name__, external_stylesheets=CSS_EXTERNO,
           suppress_callback_exceptions=True, title='E-Commerce UK · Dashboard')

app.index_string = '''<!DOCTYPE html>
<html>
    <head>
        {%metas%}
        <title>{%title%}</title>
        {%favicon%}
        {%css%}
        <style>
            /* oculto en vista normal */
            #print-view { display: none; }

            @media print {
                * { -webkit-print-color-adjust: exact !important; print-color-adjust: exact !important; }

                @page {
                    size: A4 landscape;
                    margin: 0.3cm 0.8cm 0.5cm 0.8cm;
                }

                #app-header, #tabs-bar, #filter-bar, #app-footer,
                #btn-print, .modebar-container { display: none !important; }

                #contenido-tabs { display: none !important; }
                #print-view     { display: block !important; }

                /* Contenedor de altura fija = una hoja A4 landscape.
                   overflow:hidden garantiza que nada desborde a la página siguiente. */
                .print-section {
                    page-break-after : always;
                    break-after      : page;
                    height           : 195mm;
                    overflow         : hidden;
                }
                .print-section:last-child {
                    page-break-after : avoid;
                    break-after      : avoid;
                }
            }
        </style>
    </head>
    <body>
        {%app_entry%}
        <footer>{%config%}{%scripts%}{%renderer%}</footer>
    </body>
</html>'''

select_style = {
    'background':PAL['bg3'],'color':PAL['text'],'border':f'1px solid {PAL["border"]}',
    'borderRadius':'8px','fontSize':'12.5px','fontFamily':'DM Sans, sans-serif',
    'padding':'5px 10px','cursor':'pointer',
}
_SHOW_FILTER = {'display':'flex','alignItems':'center','gap':'6px'}
_HIDE_FILTER = {'display':'none'}

app.layout = html.Div(style=ESTILOS['app'], children=[

    html.Header(id='app-header', style=ESTILOS['header'], children=[
        html.Div(style=ESTILOS['brand'], children=[
            html.Div(style=ESTILOS['dot']),
            html.Span('E-Commerce UK Retailer', style=ESTILOS['title']),
            html.Span('dic 2010 – dic 2011',    style=ESTILOS['sub']),
        ]),
        html.Button('⬇ Exportar PDF', id='btn-print', n_clicks=0, style={
            'background':alpha(PAL['azul'],0.10),'border':f'1px solid {alpha(PAL["azul"],0.40)}',
            'borderRadius':'8px','color':PAL['azul'],'cursor':'pointer',
            'fontSize':'12px','fontFamily':'DM Sans, sans-serif','fontWeight':'500',
            'padding':'7px 16px','letterSpacing':'.02em','transition':'background 0.15s',
        }),
    ]),

    html.Div(id='tabs-bar', style={'background':PAL['bg2'],'borderBottom':f'1px solid {PAL["border"]}','padding':'0 28px'}, children=[
        dcc.Tabs(id='tabs', value='tab-1', style={'border':'none'}, children=[
            dcc.Tab(label='📊 Resumen Ejecutivo',       value='tab-1', style=ESTILOS['tab_style'], selected_style=ESTILOS['tab_selected']),
            dcc.Tab(label='👥 Clientes',                value='tab-2', style=ESTILOS['tab_style'], selected_style=ESTILOS['tab_selected']),
            dcc.Tab(label='📦 Productos',               value='tab-3', style=ESTILOS['tab_style'], selected_style=ESTILOS['tab_selected']),
            dcc.Tab(label='🌍 Geografía y Operaciones', value='tab-4', style=ESTILOS['tab_style'], selected_style=ESTILOS['tab_selected']),
        ]),
    ]),

    html.Div(id='filter-bar', style={
        'display':'flex','alignItems':'center','gap':'12px','padding':'10px 28px',
        'background':PAL['bg2'],'borderBottom':f'1px solid {PAL["border"]}','minHeight':'44px',
    }, children=[
        html.Div(id='filter-seg-group', style=_HIDE_FILTER, children=[
            html.Span('Segmento:', style={**ESTILOS['filter_label'],'marginRight':'6px','whiteSpace':'nowrap'}),
            dcc.Dropdown(id='dd-segmento',
                options=[{'label':'Todos los clientes','value':'TODOS'},
                         {'label':'★ Mayorista','value':'Mayorista'},
                         {'label':'Retail','value':'Retail'}],
                value='TODOS', clearable=False, style={**select_style,'width':'175px'}),
        ]),
        html.Div(id='filter-pais-group', style=_HIDE_FILTER, children=[
            html.Span('Países:', style={**ESTILOS['filter_label'],'marginRight':'6px','whiteSpace':'nowrap'}),
            dcc.Dropdown(id='dd-paises-filtro',
                options=[{'label':'Todos (Top 15)','value':'TODOS'},
                         {'label':'★ Solo Mayoristas','value':'MAYOR'},
                         {'label':'Solo Retail Intl.','value':'RETAIL'}],
                value='TODOS', clearable=False, style={**select_style,'width':'190px'}),
        ]),
        html.Span(id='filter-hint', style={'fontSize':'10.5px','color':PAL['text3'],'fontStyle':'italic'}),
    ]),

    html.Div(id='contenido-tabs', style=ESTILOS['main']),
    html.Div(id='print-view', style={'display':'none'}),
    html.Div(id='print-dummy', style={'display':'none'}),
    dcc.Store(id='print-ready', data=0),

    html.Footer(id='app-footer',
        children='Portfolio Data Analyst · E-Commerce UK Retailer · Kaggle E-Commerce Data (carrie1) · dic 2010 – dic 2011',
        style=ESTILOS['footer']),
])

---
## 5. Callbacks — interactividad real <a id='5'></a>

Los callbacks son la diferencia clave entre un dashboard estático y uno interactivo.
Cuando el usuario cambia un filtro, Dash llama la función Python correspondiente,
que recalcula el DataFrame y devuelve el gráfico actualizado.

In [6]:
import traceback

# ── Callback: construye la vista de impresión (todas las hojas) ──────────
@app.callback(
    Output('print-view',  'children'),
    Output('print-ready', 'data'),
    Input('btn-print', 'n_clicks'),
    State('dd-segmento',    'value'),
    State('dd-paises-filtro','value'),
    prevent_initial_call=True,
)
def prepare_print(n_clicks, segmento, filtro_paises):
    seg = segmento      or 'TODOS'
    fil = filtro_paises or 'TODOS'
    content = html.Div([
        html.Div(layout_p1(pm=True),      className='print-section'),
        html.Div(layout_p2(seg, pm=True), className='print-section'),
        html.Div(layout_p3(pm=True),      className='print-section'),
        html.Div(layout_p4(fil, pm=True), className='print-section'),
    ], style={'padding':'0','margin':'0','width':'100%'})
    return content, n_clicks

# ── Clientside: espera que Plotly monte los gráficos y lanza la impresión.
#    Los SVG ya tienen width/height fijos (autosize=False) — no dependen del
#    tamaño del contenedor, así que no hay condición de carrera. ────────────
app.clientside_callback(
    """function(ready) {
        if (!ready) return '';
        setTimeout(function() {
            window.onafterprint = function() { window.location.reload(); };
            window.print();
        }, 1200);
        return '';
    }""",
    Output('print-dummy', 'children'),
    Input('print-ready', 'data'),
    prevent_initial_call=True,
)

# ── Callback principal: renderiza la tab activa ──────────────────────────
@app.callback(
    Output('contenido-tabs', 'children'),
    [Input('tabs', 'value'),
     Input('dd-segmento', 'value'),
     Input('dd-paises-filtro', 'value')]
)
def render_tab(tab, segmento, filtro_paises):
    seg = segmento      or 'TODOS'
    fil = filtro_paises or 'TODOS'
    try:
        if tab == 'tab-1': return layout_p1()
        if tab == 'tab-2': return layout_p2(seg)
        if tab == 'tab-3': return layout_p3()
        if tab == 'tab-4': return layout_p4(fil)
        return layout_p1()
    except Exception as e:
        err = traceback.format_exc()
        return html.Div([html.Pre(
            f'ERROR en {tab}:\n{err}',
            style={'color':'#f0546a','background':'#13161e','padding':'20px',
                   'borderRadius':'8px','fontSize':'12px','whiteSpace':'pre-wrap',
                   'border':'1px solid #f0546a','fontFamily':'DM Mono,monospace'}
        )])

# ── Callback secundario: muestra/oculta filtros según tab ────────────────
@app.callback(
    Output('filter-seg-group',  'style'),
    Output('filter-pais-group', 'style'),
    Output('filter-hint',       'children'),
    Input('tabs', 'value')
)
def toggle_filters(tab):
    show, hide = _SHOW_FILTER, _HIDE_FILTER
    if tab == 'tab-2': return show, hide, 'Filtra Pareto · frecuencia · Top 10 clientes'
    if tab == 'tab-4': return hide, show, 'Filtra Top 15 países y ratio de ticket'
    return hide, hide, ''

# ── Helpers ──────────────────────────────────────────────────────────────
def _pct(v, d=1): return f'{round(v, d)}%'

_HP = 175   # altura de gráficos en modo print (px)
_WP = 460   # ancho fijo en modo print — evita que Plotly mida el contenedor oculto

def _gh(pm):
    return {'height': f'{_HP}px'} if pm else {}

def _cancel_fig(pm=False):
    media = cancel_by_mes['tasa'].mean()
    cols  = [alpha(PAL['verde'],0.8) if v <= media else alpha(PAL['rojo'],0.75)
             for v in cancel_by_mes['tasa']]
    fig = go.Figure()
    fig.add_bar(x=cancel_by_mes['label'], y=cancel_by_mes['tasa'],
                marker_color=cols, marker_cornerradius=4,
                hovertemplate='<b>%{x}</b><br>Tasa: %{y:.2f}%<extra></extra>')
    fig.add_scatter(x=cancel_by_mes['label'], y=[media]*len(cancel_by_mes),
                    mode='lines', line=dict(color=alpha(PAL['text2'],0.45), width=1.5, dash='dash'),
                    hoverinfo='skip')
    fig.update_layout(**base_layout(), xaxis_tickangle=45,
                      yaxis_range=[8, 22], yaxis_ticksuffix='%')
    if pm: fig.update_layout(height=_HP, width=_WP, autosize=False)
    return fig

# ══ TAB 1 ══════════════════════════════════════════════════════════════════
def layout_p1(pm=False):
    prom     = rev_mensual['Revenue'].mean()
    mes_pico = rev_mensual.loc[rev_mensual['Revenue'].idxmax()]
    pct_h    = (rev_hora[rev_hora['Hour'].between(10,15)]['Revenue'].sum()
                / rev_hora['Revenue'].sum() * 100)

    cols_m = [PAL['azul'] if v >= 1e6 else alpha(PAL['azul'],0.40) for v in rev_mensual['Revenue']]
    fig_mensual = go.Figure()
    fig_mensual.add_bar(x=rev_mensual['label'], y=rev_mensual['Revenue'],
                        marker_color=cols_m, marker_cornerradius=5,
                        hovertemplate='<b>%{x}</b><br>£%{y:,.0f}<extra></extra>')
    fig_mensual.add_scatter(x=rev_mensual['label'], y=[prom]*len(rev_mensual),
                            mode='lines', line=dict(color=alpha(PAL['rojo'],0.65), width=1.5, dash='dash'),
                            hoverinfo='skip')
    fig_mensual.add_annotation(x=rev_mensual['label'].iloc[-1], y=prom,
                               text=f' Prom {fmt_gbp(prom)}', showarrow=False,
                               font=dict(color=PAL['text3'], size=11), xanchor='left')
    fig_mensual.update_layout(**base_layout(), yaxis_title='Revenue',
                              yaxis_tickformat='.3~s', yaxis_tickprefix='£')
    if pm: fig_mensual.update_layout(height=_HP, width=_WP, autosize=False)

    cols_h = [PAL['azul'] if 10 <= h <= 15 else alpha(PAL['azul'],0.28) for h in rev_hora['Hour']]
    fig_hora = go.Figure()
    fig_hora.add_bar(x=rev_hora['Hour'], y=rev_hora['Revenue'],
                     marker_color=cols_h, marker_cornerradius=4,
                     hovertemplate='<b>%{x}h</b><br>£%{y:,.0f}<extra></extra>')
    fig_hora.add_annotation(x=12, y=rev_hora['Revenue'].max()*0.88,
                            text=f'10–15h = {pct_h:.1f}%', showarrow=False,
                            font=dict(color=PAL['azul'], size=12, family='DM Mono'),
                            bgcolor=alpha(PAL['azul'],0.10), bordercolor=alpha(PAL['azul'],0.3), borderpad=6)
    fig_hora.update_layout(**base_layout(), xaxis_title='Hora del día',
                           xaxis_tickvals=list(range(6, 21, 2)),
                           yaxis_tickformat='.3~s', yaxis_tickprefix='£')
    if pm: fig_hora.update_layout(height=_HP, width=_WP, autosize=False)

    rev_uk   = ventas_id[ventas_id['Country']=='United Kingdom']['Revenue'].sum()
    rev_intl = ventas_id[ventas_id['Country']!='United Kingdom']['Revenue'].sum()
    rev_gest = ventas[ventas['IsGuest']]['Revenue'].sum()
    fig_seg  = go.Figure(go.Pie(
        labels=['UK', 'Internacional', 'Guest'], values=[rev_uk, rev_intl, rev_gest], hole=0.65,
        marker_colors=[alpha(PAL['azul'],0.85), alpha(PAL['verde'],0.85), alpha(PAL['ocre'],0.85)],
        insidetextfont=dict(color=PAL['text2'], size=12),
        hovertemplate='<b>%{label}</b><br>£%{value:,.0f}<br>%{percent}<extra></extra>',
    ))
    fig_seg.update_layout(**base_layout())
    fig_seg.update_layout(showlegend=True, legend=dict(font=dict(color=PAL['text2']), bgcolor='rgba(0,0,0,0)'))
    if pm: fig_seg.update_layout(height=_HP, width=_WP, autosize=False)

    fig_cancel = _cancel_fig(pm=pm)
    ratio_pico = mes_pico['Revenue'] / prom
    g = _gh(pm)
    return html.Div([
        html.Div('Resumen Ejecutivo', style={'fontSize':'20px','fontWeight':'600','color':PAL['text'],'marginBottom':'8px'}),
        html.Div([
            'El negocio facturó ', html.Strong(fmt_gbp(KPI['rev_neto'])),
            f' neto (dic-2010 – dic-2011). {mes_pico["label"]} fue el mes pico con ',
            html.Strong(f'{fmt_gbp(mes_pico["Revenue"])} — {ratio_pico:.2f}× el promedio mensual'),
            f'. El {pct_h:.1f}% del revenue ocurre entre las 10h y 15h. ',
            f'Cancelación estable: media {KPI["tasa_cancel_media"]}%, CV {KPI["cv_cancel"]}%.',
        ], style=ESTILOS['narrative']),
        html.Div(style=ESTILOS['kpi_strip'], children=[
            kpi_card('Revenue Neto',     fmt_gbp(KPI['rev_neto']),   'Ventas menos cancelaciones'),
            kpi_card('Facturas',         fmt_num(KPI['n_facturas']),  'InvoiceNo únicos'),
            kpi_card('Clientes Únicos',  fmt_num(KPI['n_clientes']),  'CustomerID identificados'),
            kpi_card('Ticket Promedio',  fmt_gbp(KPI['ticket_avg']),  'Por factura'),
            kpi_card('Tasa Cancelación', _pct(KPI['tasa_cancel_media']),
                     badge=f'CV {KPI["cv_cancel"]}% · Estable', badge_color=PAL['verde']),
            kpi_card('Costo Envíos',     _pct(KPI['pct_envios']),    fmt_gbp(KPI['rev_envios'])),
        ]),
        html.Div(style=ESTILOS['grid_2'], children=[
            card('Revenue mensual 2011', f'Ene–Nov · prom = {fmt_gbp(prom)}',
                 [dcc.Graph(figure=fig_mensual, config={'displayModeBar':False}, style=g)]),
            card('Revenue por hora del día', f'Franja 10–15h = {pct_h:.1f}% del revenue',
                 [dcc.Graph(figure=fig_hora, config={'displayModeBar':False}, style=g)]),
        ]),
        html.Div(style=ESTILOS['grid_2'], children=[
            card('Composición del revenue', 'UK identificado · Internacional · Guest (anónimo)',
                 [dcc.Graph(figure=fig_seg, config={'displayModeBar':False}, style=g)]),
            card('Tasa de cancelación mensual', f'Verde ≤ media {KPI["tasa_cancel_media"]}% · Rojo > media',
                 [dcc.Graph(figure=fig_cancel, config={'displayModeBar':False}, style=g)]),
        ]),
    ])

# ══ TAB 2 ══════════════════════════════════════════════════════════════════
def layout_p2(segmento='TODOS', pm=False):
    df_seg   = ventas_id if segmento == 'TODOS' else ventas_id[ventas_id['Segment']==segmento]
    t_mayor  = ventas_id[ventas_id['Segment']=='Mayorista'].groupby('InvoiceNo')['Revenue'].sum().mean()
    t_retail = ventas_id[ventas_id['Segment']=='Retail'].groupby('InvoiceNo')['Revenue'].sum().mean()
    t_guest  = ventas[ventas['IsGuest']].groupby('InvoiceNo')['Revenue'].sum().mean()
    rev_id   = ventas_id['Revenue'].sum()
    pct_id   = rev_id / ventas['Revenue'].sum() * 100

    n_ped_all = ventas_id.groupby('CustomerID')['InvoiceNo'].nunique()
    t_cid_all = (ventas_id.groupby(['CustomerID','InvoiceNo'])['Revenue'].sum()
                 .groupby('CustomerID').mean())
    idx  = n_ped_all.index.intersection(t_cid_all.index)
    r_df = pd.DataFrame({'n': n_ped_all[idx], 't': t_cid_all[idx]}).dropna()
    corr = np.corrcoef(r_df['n'].values, r_df['t'].values)[0,1] if len(r_df) > 1 else 0

    rc = (df_seg.groupby('CustomerID')['Revenue'].sum()
          .sort_values(ascending=False).reset_index())
    rc['acum'] = rc['Revenue'].cumsum() / rc['Revenue'].sum() * 100
    rc['rank'] = np.arange(1, len(rc)+1) / len(rc) * 100
    top20_n = int(len(rc) * 0.20)
    pct20   = rc.head(top20_n)['Revenue'].sum() / rc['Revenue'].sum() * 100 if len(rc) > 0 else 0

    fig_pareto = go.Figure()
    fig_pareto.add_scatter(x=rc['rank'], y=rc['acum'], mode='lines', fill='tozeroy',
                           line=dict(color=PAL['azul'], width=2.5), fillcolor=alpha(PAL['azul'],0.07),
                           hovertemplate='Top %{x:.1f}% clientes → %{y:.1f}% del revenue<extra></extra>')
    fig_pareto.add_vline(x=20, line_dash='dot', line_color=alpha(PAL['text3'],0.5))
    fig_pareto.add_hline(y=pct20, line_dash='dash', line_color=alpha(PAL['rojo'],0.5))
    fig_pareto.add_annotation(x=22, y=pct20+3, text=f'Top 20% → {pct20:.1f}%',
                               font=dict(color=PAL['rojo'], size=11), showarrow=False)
    fig_pareto.update_layout(**base_layout(), xaxis_title='% acumulado de clientes',
                              yaxis_title='% acumulado del revenue',
                              xaxis_ticksuffix='%', yaxis_ticksuffix='%', yaxis_range=[0,100])
    if pm: fig_pareto.update_layout(height=_HP, width=_WP, autosize=False)

    fig_ticket = go.Figure(go.Bar(
        x=[t_mayor, t_guest, t_retail], y=['Mayorista', 'Guest', 'Retail'], orientation='h',
        marker_color=[alpha(PAL['ocre'],0.85), alpha(PAL['violeta'],0.85), alpha(PAL['azul'],0.85)],
        marker_cornerradius=5, hovertemplate='<b>%{y}</b><br>Ticket: £%{x:,.0f}<extra></extra>',
    ))
    fig_ticket.update_layout(**base_layout(), xaxis_tickformat='.3~s', xaxis_tickprefix='£')
    if pm: fig_ticket.update_layout(height=_HP, width=_WP, autosize=False)

    n_ped_seg = df_seg.groupby('CustomerID')['InvoiceNo'].nunique()
    freq_vals = [int((n_ped_seg==1).sum()), int(((n_ped_seg>=2)&(n_ped_seg<=5)).sum()),
                 int(((n_ped_seg>=6)&(n_ped_seg<=20)).sum()), int((n_ped_seg>20).sum())]
    fig_freq = go.Figure(go.Bar(
        x=['1 pedido','2-5','6-20','21+'], y=freq_vals,
        marker_color=[alpha(PAL['azul'],0.9), alpha(PAL['verde'],0.8),
                      alpha(PAL['ocre'],0.75), alpha(PAL['rojo'],0.7)],
        marker_cornerradius=5, hovertemplate='<b>%{x}</b><br>%{y:,} clientes<extra></extra>',
    ))
    fig_freq.update_layout(**base_layout(), yaxis_title='N° de clientes', xaxis_title='Pedidos por cliente')
    if pm: fig_freq.update_layout(height=_HP, width=_WP, autosize=False)

    top10 = (df_seg.groupby('CustomerID')['Revenue'].sum()
             .sort_values(ascending=False).head(10).reset_index())
    top10['lbl'] = top10['CustomerID'].apply(lambda c: f'CID {int(c)}')
    top10['col'] = top10['CustomerID'].apply(
        lambda c: alpha(PAL['ocre'],0.85) if c in MAYOR_CIDS else alpha(PAL['verde'],0.75))
    fig_top10 = go.Figure(go.Bar(
        x=top10['Revenue'].values[::-1], y=top10['lbl'].values[::-1], orientation='h',
        marker_color=top10['col'].values[::-1].tolist(), marker_cornerradius=4,
        hovertemplate='<b>%{y}</b><br>Revenue: £%{x:,.0f}<extra></extra>',
    ))
    fig_top10.update_layout(**base_layout(), xaxis_tickformat='.3~s', xaxis_tickprefix='£')
    if pm: fig_top10.update_layout(height=_HP, width=_WP, autosize=False)

    n_seg = len(rc)
    g = _gh(pm)
    return html.Div([
        html.Div('Análisis de Clientes', style={'fontSize':'20px','fontWeight':'600','color':PAL['text'],'marginBottom':'8px'}),
        html.Div([
            html.Strong(f'Top 20% ({top20_n:,} clientes) genera el {pct20:.1f}% del revenue. '),
            f'Los {KPI["n_mayor_cids"]} mayoristas tienen ticket ', html.Strong(fmt_gbp(t_mayor)),
            f' — {t_mayor/t_retail:.1f}× el ticket retail ({fmt_gbp(t_retail)}). ',
            'Guest gasta ', html.Strong(f'{t_guest/t_retail:.1f}×'), ' más que retail por factura. ',
            f'Correlación frecuencia–ticket: r = {corr:.3f} (sin relación lineal).',
        ], style=ESTILOS['narrative']),
        html.Div(style=ESTILOS['kpi_strip'], children=[
            kpi_card('Pareto Top 20%',    _pct(pct20),       f'{top20_n:,} de {n_seg:,} clientes'),
            kpi_card('Ticket Mayorista',  fmt_gbp(t_mayor),  badge=f'{KPI["n_mayor_cids"]} clientes', badge_color=PAL['ocre']),
            kpi_card('Ticket Retail',     fmt_gbp(t_retail), badge=f'{len(RETAIL_CIDS):,} clientes', badge_color=PAL['azul']),
            kpi_card('Ticket Guest',      fmt_gbp(t_guest),  badge=f'{t_guest/t_retail:.1f}× vs retail', badge_color=PAL['ocre']),
            kpi_card('Corr. Frec/Ticket', f'{corr:.3f}',     badge='Sin relación', badge_color=PAL['verde']),
            kpi_card('Rev. Identificado', _pct(pct_id),      f'{100-pct_id:.1f}% anónimo (Guest)'),
        ]),
        html.Div(style=ESTILOS['grid_2_1'], children=[
            card('Curva de Pareto — clientes identificados',
                 f'% acumulado clientes vs revenue · filtro: {segmento}',
                 [dcc.Graph(figure=fig_pareto, config={'displayModeBar':False}, style=g)]),
            card('Ticket promedio por segmento', 'Revenue promedio por factura · Mayorista / Guest / Retail',
                 [dcc.Graph(figure=fig_ticket, config={'displayModeBar':False}, style=g)]),
        ]),
        html.Div(style=ESTILOS['grid_2'], children=[
            card('Frecuencia de compra por cliente',
                 f'N° de pedidos por cliente en el período · filtro: {segmento}',
                 [dcc.Graph(figure=fig_freq, config={'displayModeBar':False}, style=g)]),
            card('Top 10 clientes por revenue',
                 f'Naranja = mayorista · Verde = retail · filtro: {segmento}',
                 [dcc.Graph(figure=fig_top10, config={'displayModeBar':False}, style=g)]),
        ]),
    ])

# ══ TAB 3 ══════════════════════════════════════════════════════════════════
def layout_p3(pm=False):
    pct10  = sc_rev.head(10)['pct'].sum()
    pct50  = sc_rev.head(50)['pct'].sum()
    pct100 = sc_rev.head(100)['pct'].sum()
    n_sc_sin     = inv_excl['StockCode'].nunique()
    total_bajas  = bajas_m['n_bajas'].sum()
    bajas_eneabr = bajas_m[(bajas_m['Year']==2011) & (bajas_m['Month'].isin([1,2,3,4]))]['n_bajas'].sum()
    pct_eneabr   = round(bajas_eneabr / total_bajas * 100, 1) if total_bajas > 0 else 0

    fig_conc = go.Figure()
    fig_conc.add_scatter(x=conc_df['n'], y=conc_df['pct'], mode='lines+markers', fill='tozeroy',
                         line=dict(color=PAL['verde'], width=2.5), fillcolor=alpha(PAL['verde'],0.06),
                         marker=dict(size=5, color=PAL['verde']),
                         hovertemplate='Top %{x} SKUs → %{y:.1f}% del revenue<extra></extra>')
    for n_m, c_m, l_m in [(50,PAL['rojo'],'50'),(100,PAL['ocre'],'100'),(200,PAL['text3'],'200')]:
        row = conc_df[conc_df['n']==n_m]
        if len(row):
            pv = row['pct'].values[0]
            fig_conc.add_annotation(x=n_m, y=pv+3, text=f'Top {l_m}: {pv:.1f}%',
                                    font=dict(color=c_m, size=10),
                                    showarrow=True, arrowcolor=c_m, arrowsize=0.7, ax=30, ay=-20)
    fig_conc.update_layout(**base_layout(), xaxis_title='N° de SKUs',
                           yaxis_title='% acumulado del revenue', yaxis_range=[0,100], yaxis_ticksuffix='%')
    if pm: fig_conc.update_layout(height=_HP, width=_WP, autosize=False)

    t10     = sc_rev.head(10).copy()
    desc10  = [str(d)[:28]+'…' if len(str(d))>28 else str(d) for d in t10['Description']]
    cols_sc = [alpha(PAL['azul'],0.90) if i<3 else alpha(PAL['azul'],0.55) for i in range(10)]
    fig_top10sc = go.Figure(go.Bar(
        x=t10['Revenue'].values[::-1], y=desc10[::-1], orientation='h',
        marker_color=cols_sc[::-1], marker_cornerradius=4,
        customdata=t10['StockCode'].values[::-1],
        hovertemplate='<b>%{customdata}</b><br>%{y}<br>Revenue: £%{x:,.0f}<extra></extra>',
    ))
    fig_top10sc.update_layout(**base_layout(), xaxis_tickformat='.3~s', xaxis_tickprefix='£',
                              yaxis_tickfont=dict(size=11))
    if pm: fig_top10sc.update_layout(height=_HP, width=_WP, autosize=False)

    n_rows    = 6 if pm else 20
    desc_len  = 20 if pm else 34
    tabla_data = top20_sc.head(n_rows)

    _td = lambda extra: {
        'padding': '4px 8px' if pm else '5px 8px',
        'whiteSpace': 'nowrap',
        'overflow': 'hidden',
        'textOverflow': 'ellipsis',
        **extra
    }

    rows = []
    for _, row in tabla_data.iterrows():
        bw = min(int(row['pct'] / 1.8 * 100), 120)
        rows.append(html.Tr([
            html.Td(row['StockCode'],
                    style=_td({'fontFamily':'DM Mono,monospace','color':PAL['azul'],'fontSize':'12px','maxWidth':'70px'})),
            html.Td(str(row['Description'])[:desc_len],
                    style=_td({'color':PAL['text'],'fontSize':'12px','maxWidth':'160px'})),
            html.Td(f'£{row["Revenue"]:,.0f}',
                    style=_td({'fontFamily':'DM Mono,monospace','color':PAL['text2'],'fontSize':'12px','maxWidth':'80px'})),
            html.Td([f'{row["pct"]:.2f}%',
                     html.Span(style={'display':'inline-block','height':'5px','width':f'{bw}px',
                                      'background':PAL['azul'],'opacity':'.5','borderRadius':'3px',
                                      'marginLeft':'6px','verticalAlign':'middle'})],
                    style=_td({'fontSize':'12px','color':PAL['text2'],'maxWidth':'120px'})),
            html.Td(f'{row["acum"]:.2f}%',
                    style=_td({'fontFamily':'DM Mono,monospace','color':PAL['text3'],'fontSize':'11px','maxWidth':'60px'})),
        ], style={'borderBottom':f'1px solid {PAL["border"]}'}))

    tabla_sub = 'SC · Descripción · Revenue · % · Acum.' + (' · Top 6' if pm else ' · Top 20')
    tabla = html.Table([
        html.Thead(html.Tr([html.Th(h, style={'color':PAL['text3'],'fontSize':'10.5px','padding':'5px 8px',
                                              'textTransform':'uppercase','fontWeight':'600','whiteSpace':'nowrap'})
                            for h in ['SC','Descripción','Revenue','%','Acum.']],
                           style={'borderBottom':f'1px solid rgba(255,255,255,0.14)'})),
        html.Tbody(rows),
    ], style={'width':'100%','borderCollapse':'collapse','tableLayout':'fixed'})

    cols_b  = [alpha(PAL['rojo'],0.85) if v >= 13 else alpha(PAL['rojo'],0.35) for v in bajas_m['n_bajas']]
    fig_bajas = go.Figure(go.Bar(x=bajas_m['label'], y=bajas_m['n_bajas'],
                                  marker_color=cols_b, marker_cornerradius=4,
                                  hovertemplate='<b>%{x}</b><br>%{y} bajas<extra></extra>'))
    fig_bajas.update_layout(**base_layout(), yaxis_title='N° de bajas', xaxis_tickangle=45)
    if pm: fig_bajas.update_layout(height=_HP, width=_WP, autosize=False)

    g = _gh(pm)
    return html.Div([
        html.Div('Análisis de Productos', style={'fontSize':'20px','fontWeight':'600','color':PAL['text'],'marginBottom':'8px'}),
        html.Div([
            'El catálogo tiene ', html.Strong(f'{KPI["n_productos"]:,} SKUs activos. '),
            f'Solo 50 SKUs (1.3%) generan el ', html.Strong(f'{pct50:.1f}% del revenue'),
            f'. Top 10 = {pct10:.1f}%, Top 100 = {pct100:.1f}%. ',
            f'Los {n_sc_sin} SKUs de inventario sin ventas concentraron el ',
            html.Strong(f'{pct_eneabr}% de sus bajas en ene-abr 2011'), ' — limpieza puntual.',
        ], style=ESTILOS['narrative']),
        html.Div(style=ESTILOS['kpi_strip'], children=[
            kpi_card('SKUs Activos',       fmt_num(KPI['n_productos']), 'StockCodes con ventas'),
            kpi_card('Top 10 SKUs',        _pct(pct10),  'Del revenue total'),
            kpi_card('Top 50 SKUs',        _pct(pct50),  '1.3% del catálogo'),
            kpi_card('Top 100 SKUs',       _pct(pct100), 'Del revenue total'),
            kpi_card('SKUs Sin Venta',     str(n_sc_sin), badge='Dados de baja', badge_color=PAL['ocre']),
            kpi_card('Bajas ene-abr 2011', _pct(pct_eneabr), 'En ene-abr 2011'),
        ]),
        html.Div(style=ESTILOS['grid_1_2'], children=[
            card('Concentración del catálogo', '% revenue acumulado por N° de SKUs',
                 [dcc.Graph(figure=fig_conc, config={'displayModeBar':False}, style=g)]),
            card('Top 10 productos por revenue', 'Mayor revenue arriba · hover para StockCode completo',
                 [dcc.Graph(figure=fig_top10sc, config={'displayModeBar':False}, style=g)]),
        ]),
        html.Div(style=ESTILOS['grid_2_1'], children=[
            card('Top 20 productos — detalle', tabla_sub, [tabla]),
            card('Bajas de inventario por mes', f'{n_sc_sin} SKUs sin ventas · pico ene-abr 2011',
                 [dcc.Graph(figure=fig_bajas, config={'displayModeBar':False}, style=g)]),
        ]),
    ])

# ══ TAB 4 ══════════════════════════════════════════════════════════════════
def layout_p4(filtro='TODOS', pm=False):
    if filtro == 'MAYOR':    ps = pais_stats[pais_stats['IsMayor']].head(15).copy()
    elif filtro == 'RETAIL': ps = pais_stats[~pais_stats['IsMayor']].head(15).copy()
    else:                    ps = pais_stats.head(15).copy()

    labels_p = [('★ ' if r['IsMayor'] else '') + r['Country'] for _, r in ps.iterrows()]
    cols_p   = [alpha(PAL['ocre'],0.85) if r['IsMayor'] else alpha(PAL['azul'],0.70) for _, r in ps.iterrows()]
    cols_r   = [alpha(PAL['ocre'],0.85) if r['IsMayor'] else alpha(PAL['verde'],0.70) for _, r in ps.iterrows()]

    fig_pais = go.Figure(go.Bar(
        x=ps['Revenue'].values[::-1], y=labels_p[::-1], orientation='h',
        marker_color=cols_p[::-1], marker_cornerradius=4, customdata=ps['n_cli'].values[::-1],
        hovertemplate='<b>%{y}</b><br>Revenue: £%{x:,.0f}<br>Clientes: %{customdata}<extra></extra>',
    ))
    fig_pais.update_layout(**base_layout(), xaxis_tickformat='.3~s', xaxis_tickprefix='£',
                           yaxis_tickfont=dict(size=11))
    if pm: fig_pais.update_layout(height=_HP, width=_WP, autosize=False)

    fig_ratio = go.Figure(go.Bar(
        x=ps['ratio'].values[::-1], y=labels_p[::-1], orientation='h',
        marker_color=cols_r[::-1], marker_cornerradius=4,
        hovertemplate='<b>%{y}</b><br>%{x:.2f}× ticket UK<extra></extra>',
    ))
    fig_ratio.add_vline(x=3, line_dash='dot', line_color=alpha(PAL['rojo'],0.55), line_width=1.5)
    fig_ratio.add_annotation(x=3.1, y=-0.5, text='Umbral mayorista (3×)',
                              font=dict(color=PAL['rojo'], size=10), showarrow=False, xanchor='left')
    fig_ratio.update_layout(**base_layout(), xaxis_ticksuffix='×', yaxis_tickfont=dict(size=11))
    if pm: fig_ratio.update_layout(height=_HP, width=_WP, autosize=False)

    fig_cancel = _cancel_fig(pm=pm)

    rev_uk_ret   = ventas_id[(ventas_id['Country']=='United Kingdom') & (~ventas_id['CustomerID'].isin(MAYOR_CIDS))]['Revenue'].sum()
    rev_uk_may   = ventas_id[(ventas_id['Country']=='United Kingdom') &  (ventas_id['CustomerID'].isin(MAYOR_CIDS))]['Revenue'].sum()
    rev_uk_total = rev_uk_ret + rev_uk_may
    rev_intl_ret = ventas_id[(ventas_id['Country']!='United Kingdom') & (~ventas_id['CustomerID'].isin(MAYOR_CIDS))]['Revenue'].sum()
    rev_intl_may = ventas_id[(ventas_id['Country']!='United Kingdom') &  (ventas_id['CustomerID'].isin(MAYOR_CIDS))]['Revenue'].sum()
    rev_guest    = ventas[ventas['IsGuest']]['Revenue'].sum()

    fig_origen = go.Figure(go.Pie(
        labels=['UK Retail', 'UK Mayorista', 'Intl. Retail', 'Intl. Mayorista', 'Guest'],
        values=[rev_uk_ret, rev_uk_may, rev_intl_ret, rev_intl_may, rev_guest], hole=0.60,
        marker_colors=[alpha(PAL['azul'],0.85), alpha(PAL['violeta'],0.80),
                       alpha(PAL['verde'],0.85), alpha(PAL['ocre'],0.85), alpha(PAL['text3'],0.80)],
        insidetextfont=dict(color=PAL['text2'], size=11),
        hovertemplate='<b>%{label}</b><br>£%{value:,.0f}<br>%{percent}<extra></extra>',
    ))
    fig_origen.update_layout(**base_layout())
    fig_origen.update_layout(showlegend=True,
                             legend=dict(font=dict(color=PAL['text2']), bgcolor='rgba(0,0,0,0)',
                                         orientation='h', y=-0.15))
    if pm: fig_origen.update_layout(height=_HP, width=_WP, autosize=False)

    n_cli_uk    = ventas_id[ventas_id['Country']=='United Kingdom']['CustomerID'].nunique()
    n_cli_intl  = ventas_id[ventas_id['Country']!='United Kingdom']['CustomerID'].nunique()
    rev_intl    = rev_intl_ret + rev_intl_may
    rev_uk_pc   = rev_uk_total / n_cli_uk   if n_cli_uk   > 0 else 0
    rev_intl_pc = rev_intl    / n_cli_intl  if n_cli_intl > 0 else 0
    ratio_intl  = round(rev_intl_pc / rev_uk_pc, 2) if rev_uk_pc > 0 else 0
    pct_uk      = round(rev_uk_total / ventas_id['Revenue'].sum() * 100, 1)

    non_uk = pais_stats[pais_stats['Country']!='United Kingdom'].sort_values('ratio', ascending=False)
    if len(non_uk):
        top_r = non_uk.iloc[0]; top_r_txt = f'{top_r["ratio"]:.1f}× UK'; top_r_country = top_r['Country']
    else:
        top_r = {'ratio': 0};   top_r_txt = '—'; top_r_country = '—'

    n_mayor_p    = int(pais_stats['IsMayor'].sum())
    pct_cli_intl = round(n_cli_intl / (n_cli_uk + n_cli_intl) * 100, 1) if (n_cli_uk+n_cli_intl) > 0 else 0

    g = _gh(pm)
    return html.Div([
        html.Div('Geografía y Operaciones', style={'fontSize':'20px','fontWeight':'600','color':PAL['text'],'marginBottom':'8px'}),
        html.Div([
            html.Strong(f'{n_mayor_p} países'), f' operan como mayoristas con tickets ',
            html.Strong(f'3–{top_r["ratio"]:.1f}× superiores al baseline UK'),
            f'. Los clientes internacionales son el {pct_cli_intl}% de la base pero generan ',
            html.Strong(f'{ratio_intl:.2f}× más revenue'), f' per capita que los clientes UK. ',
            f'Cancelación: media {KPI["tasa_cancel_media"]}%, CV {KPI["cv_cancel"]}% (estable).',
        ], style=ESTILOS['narrative']),
        html.Div(style=ESTILOS['kpi_strip'], children=[
            kpi_card('Revenue UK',         _pct(pct_uk),    f'{fmt_gbp(rev_uk_total)} · {fmt_num(n_cli_uk)} clientes'),
            kpi_card('Países Activos',     str(KPI['n_paises']), f'{n_mayor_p} mayoristas intl.'),
            kpi_card('Ticket máx. (país)', top_r_txt,       badge=top_r_country, badge_color=PAL['ocre']),
            kpi_card('Intl Rev/Cliente',   f'{ratio_intl:.2f}×', 'vs cliente UK'),
            kpi_card('Cancelaciones',      _pct(KPI['tasa_cancel_media']), badge=f'CV {KPI["cv_cancel"]}%', badge_color=PAL['verde']),
            kpi_card('Envíos / Revenue',   _pct(KPI['pct_envios']), fmt_gbp(KPI['rev_envios'])),
        ]),
        html.Div(style=ESTILOS['grid_2'], children=[
            card('Top 15 países por revenue', f'★ = mayorista (ticket > 3× UK) · filtro: {filtro}',
                 [dcc.Graph(figure=fig_pais, config={'displayModeBar':False}, style=g)]),
            card('Ratio ticket promedio vs UK', 'Línea roja = umbral mayorista (3×)',
                 [dcc.Graph(figure=fig_ratio, config={'displayModeBar':False}, style=g)]),
        ]),
        html.Div(style=ESTILOS['grid_2'], children=[
            card('Tasa de cancelación mensual', f'Verde ≤ media {KPI["tasa_cancel_media"]}% · Rojo > media',
                 [dcc.Graph(figure=fig_cancel, config={'displayModeBar':False}, style=g)]),
            card('Composición del revenue por origen',
                 'UK Retail · UK Mayorista · Intl. Retail · Intl. Mayorista · Guest',
                 [dcc.Graph(figure=fig_origen, config={'displayModeBar':False}, style=g)]),
        ]),
    ])


---
## 6. Correr la app <a id='6'></a>

Ejecutá esta celda. La app queda corriendo en `http://localhost:8050`.  


In [7]:
print('Abrí tu browser en: http://localhost:8050')
app.run(debug=False, port=8050, jupyter_mode='inline')

Abrí tu browser en: http://localhost:8050
